In [1]:
# ── CELL 1: Import ───────────────────────────────
import requests
import json
import tensorflow as tf
import base64

2026-06-02 04:38:40.315965: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-02 04:38:41.105663: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-02 04:38:41.111221: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/workspaces/rahadianivan09-mlpipeline/tfx-env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core pa

In [2]:
# ── CELL 2: Sample Data ──────────────────────────
# Contoh data nasabah untuk prediksi
samples = [
    {"age": 58, "job": "management", "marital": "married", "education": "tertiary",
     "default": "no", "balance": 2143, "housing": "yes", "loan": "no",
     "contact": "unknown", "day": 5, "month": "may", "duration": 261,
     "campaign": 1, "pdays": -1, "previous": 0, "poutcome": "unknown"},
    {"age": 44, "job": "technician", "marital": "single", "education": "secondary",
     "default": "no", "balance": 29, "housing": "yes", "loan": "no",
     "contact": "unknown", "day": 5, "month": "may", "duration": 151,
     "campaign": 1, "pdays": -1, "previous": 0, "poutcome": "unknown"},
]

In [3]:
# ── CELL 3: Fungsi Prediksi ──────────────────────
def create_example(sample):
    feature = {}
    int_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
    str_features = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

    for f in int_features:
        feature[f] = tf.train.Feature(int64_list=tf.train.Int64List(value=[sample[f]]))
    for f in str_features:
        feature[f] = tf.train.Feature(bytes_list=tf.train.BytesList(value=[sample[f].encode()]))

    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return base64.b64encode(example.SerializeToString()).decode()

def predict_deposit(sample):
    url = "http://localhost:8501/v1/models/bank-deposit-model:predict"
    payload = json.dumps({
        "signature_name": "serving_default",
        "instances": [{"examples": {"b64": create_example(sample)}}]
    })
    response = requests.post(url, data=payload)
    result = response.json()
    score = result["predictions"][0][0]
    label = "Akan Deposit ✅" if score > 0.5 else "Tidak Deposit ❌"
    return score, label

In [4]:
# ── CELL 4: Jalankan Prediksi ────────────────────
for i, sample in enumerate(samples):
    score, label = predict_deposit(sample)
    print(f"Nasabah {i+1}")
    print(f"  Age     : {sample['age']}, Job: {sample['job']}")
    print(f"  Score   : {score:.4f}")
    print(f"  Prediksi: {label}")
    print("-" * 50)

Nasabah 1
  Age     : 58, Job: management
  Score   : 0.3379
  Prediksi: Tidak Deposit ❌
--------------------------------------------------
Nasabah 2
  Age     : 44, Job: technician
  Score   : 0.2917
  Prediksi: Tidak Deposit ❌
--------------------------------------------------
